# claudeknows: learn water bottle, keys and shoes

Downloads internet photos (Google Open Images), adds your own photos/videos from
Google Drive if you have any, trains a small pretrained network, and shows the
accuracy per object and the photos it got wrong.

**Before you start:** Runtime -> Change runtime type -> choose a GPU (e.g. T4).
Then Runtime -> Run all. Takes roughly 15-25 minutes (most of it scanning Open
Images' big label files).

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())

## 1. Get the code

In [ ]:
REPO = "https://github.com/alexg-sketch/claudeknows"
BRANCH = "main"  # change to a claude/... branch to try unmerged work
import os
if not os.path.isdir("claudeknows_repo"):
    !git clone -b $BRANCH $REPO claudeknows_repo
%cd claudeknows_repo
!pip install -q -r requirements.txt
!ffmpeg -version | head -1

## 2. Internet photos

Water bottles, keys and shoes from Open Images (CC BY; credits are written to
`data/objects/web/CREDITS.md`). Photos are cropped to the object where the
dataset has a box. Up to 300 per object; some key photos no longer exist online,
so expect fewer keys.

In [ ]:
!python -m claudeknows.openimages --out data/objects --cap 300

## 3. Your own photos and videos (optional)

If Google Drive has `claudeknows_photos/water_bottle`, `keys` and `shoes`
folders, their photos and videos are added (videos become photos, about 2 per
second, blurry ones skipped). Missing or empty folders are fine: that object
just uses internet photos. Your files stay in your Colab session and are never
uploaded to GitHub.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/claudeknows_photos"
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Google Drive not connected (that is OK):", e)
if os.path.isdir(DRIVE_DIR):
    !python -m claudeknows.objects "$DRIVE_DIR" --root data/objects
else:
    print("No claudeknows_photos folder found: using internet photos only.")

## 4. Look at the photos it will learn from

In [ ]:
import glob, random
import matplotlib.pyplot as plt
from PIL import Image
from claudeknows.objects import OBJECT_NAMES

def show(paths, title, n=6):
    paths = random.sample(paths, min(n, len(paths)))
    fig, axes = plt.subplots(1, max(len(paths), 1), figsize=(2.4 * max(len(paths), 1), 2.6), squeeze=False)
    for ax in axes[0]:
        ax.axis("off")
    for ax, p in zip(axes[0], paths):
        ax.imshow(Image.open(p)); ax.set_title(os.path.basename(p)[:14], fontsize=7)
    fig.suptitle(title); plt.show()

for obj in OBJECT_NAMES:
    for source, label in (("web", "internet"), ("own", "your footage")):
        files = glob.glob(f"data/objects/{source}/{obj}/*.jpg")
        print(f"{obj} ({label}): {len(files)} photos")
        if files:
            show(files, f"{obj} - {label}")

## 5. Train

First only the new last layer, then the whole network. Test photos are chosen
**by video** (never by single frame), so frames from one video are never in both
training and test.

In [ ]:
EPOCHS = 8
!python -m claudeknows.train_objects --root data/objects --epochs $EPOCHS --checkpoint objects.pt

## 6. How good is it?

Accuracy overall, per object, and per source (internet photos vs your footage),
plus the confusion matrix (rows = the true object, columns = what it guessed).

In [ ]:
from claudeknows.train_objects import evaluate_objects
result = evaluate_objects("objects.pt", "data/objects", num_workers=2)

## 7. The photos it got wrong

In [ ]:
if result and result["wrong"]:
    for s, guess in result["wrong"][:12]:
        print(f"true: {OBJECT_NAMES[s.label]:<13} guessed: {guess:<13} ({s.source}) {s.path.name}")
    show([str(s.path) for s, _ in result["wrong"]], "wrong guesses", n=8)
else:
    print("Nothing wrong in the test photos (or no test photos yet).")

## 8. Try it on one image (upload any picture)

In [ ]:
from google.colab import files
up = files.upload()
for name in up:
    !python -m claudeknows.predict "$name" --checkpoint objects.pt

## 9. Save the model

Download it, or copy it to Google Drive so it survives the session.

In [ ]:
from google.colab import files
files.download("objects.pt")
# Or: !cp objects.pt /content/drive/MyDrive/